<a href="https://colab.research.google.com/github/yeshaa23/Project-A-Kelompok-4-Pertamina-PBAGenap/blob/main/Notebook/2_Prepocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================
from google.colab import drive

# Menghubungkan Google Drive dengan Colab
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# MEMUAT DATASET DARI GOOGLE DRIVE
# ============================================================
import pandas as pd

# Ganti dengan path file yang sesuai di Google Drive Anda
file_path = '/content/drive/MyDrive/ProjectA-PBA/hasil_scraping_berita.csv'  # Ganti dengan path yang tepat

# Memuat data dari file CSV
df = pd.read_csv(file_path)

# Tampilkan beberapa baris pertama untuk memastikan data dimuat dengan benar
df.head()

,Link,Judul,Isi Berita,Status,Tag,Sentimen,Penerbit
0,https://money.kompas.com/image/2017/10/03/1815...,Foto : CSR Pertamina Lubricants Kini Fokus ke ...,Diskusi mengenai CSR di industri pelumas berta...,success,Ekonomi,Netral,money.kompas.com
1,https://money.kompas.com/read/2016/06/06/14280...,"Gandeng Dua Bank, Pertamina Lubricants Yakin P...","JAKARTA, KOMPAS.com - Anak perusahaan Pertamin...",success,Kecelakaan Kerja,Positif,money.kompas.com
2,https://money.kompas.com/image/2017/08/03/0900...,"Foto : Cegah Kepunahan, Pertamina Lestarikan T...",Pelepasan Indukan Tuntong Laut oleh tim konser...,success,Bisnis,Positif,money.kompas.com
3,https://money.kompas.com/read/2016/04/11/17345...,Pertamina Menaikkan Ongkos Angkut Minyak Penam...,"BOJONEGORO, KOMPAS.com - Pemerintah terus meng...",success,Migas,Positif,money.kompas.com
4,https://money.kompas.com/image/2017/11/27/1656...,Foto : Ini Dua Fokus CSR Pertamina Patra Niaga...,Direktur Operasional PT Pertamina Patra Niaga ...,success,Akademik,Positif,money.kompas.com


In [ ]:
# ============================================================
# IMPORT LIBRARY YANG DIBUTUHKAN
# ============================================================
!pip install Sastrawi
import re
import os
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from textblob import TextBlob
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.drawing.image import Image as XLImage
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Mengabaikan peringatan
warnings.filterwarnings("ignore")

In [ ]:
# ============================================================
# INISIALISASI NLP: Stopword dan Stemmer
# ============================================================
stemmer_factory = StemmerFactory()
stemmer         = stemmer_factory.create_stemmer()

sw_factory   = StopWordRemoverFactory()
STOPWORDS_ID = set(sw_factory.get_stop_words()) | {
    # Slang & kata umum tidak informatif
    "yg", "nya", "juga", "bisa", "itu", "ini", "ada", "jadi", "dari", "ke", "di", "dan", "atau", "dengan",
    "untuk", "pada", "sudah", "akan", "telah", "saat", "karena", "namun", "tapi", "kalau", "bila", "jika",
    "sehingga", "agar", "kami", "kita", "mereka", "dia", "ia", "anda", "saya", "kamu", "belum", "pun", "pula",
    "lagi", "kini", "hal", "kata", "ujar", "tutur", "tambah", "ungkap", "terang", "sebut", "jelaskan", "menyebut",
    "menurut", "berdasarkan", "yakni", "yaitu", "antara", "lain", "berbagai", "beberapa",
    # Noise scraping
    "http", "https", "www", "com", "id", "co", "go", "baca", "juga", "copyright", "editor", "pewarta",
    "related", "artikel", "berita", "read", "more", "click", "foto", "ilustrasi", "dok", "dokumentasi",
    "caption", "sumber", "gambar",
    # Kata sangat umum di berita
    "tahun", "hari", "bulan", "persen", "rp", "juta", "miliar", "triliun", "pt", "tbk", "persero", "presiden",
    "direktur", "menteri", "kepala",
}

# ============================================================
# DEFINISIKAN FUNGSI PENGOLAHAN TEKS
# ============================================================
def to_uppercase(text: str) -> str:
    return str(text).upper()

def to_lowercase(text: str) -> str:
    return str(text).lower()

def clean_text(text: str) -> str:
    """Membersihkan teks: menghapus URL, HTML, angka, simbol, dan kata-kata pendek."""
    text = re.sub(r"http\S+|www\S+", " ", text)        # Hapus URL
    text = re.sub(r"<[^>]+>", " ", text)               # Hapus tag HTML
    text = re.sub(r"[^\w\s]", " ", text)               # Hapus tanda baca & simbol
    text = re.sub(r"\d+", " ", text)                   # Hapus angka
    text = re.sub(r"\b\w{1,2}\b", " ", text)           # Hapus kata ≤2 karakter
    text = re.sub(r"\s+", " ", text).strip()           # Normalisasi spasi
    return text

def remove_news_boilerplate(text: str) -> str:
    text = re.sub(r'\b(kompas|cnn|detik|tempo|tribunnews|kontan|bisnis)\b', ' ', text)
    text = re.sub(r'\b(com|co|id|news|finance|money|otomotif)\b', ' ', text)
    text = re.sub(r'\b(jakarta|indonesia)\b', ' ', text)
    text = re.sub(r'\b(pt|persero|tbk)\b', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def remove_stopwords(text: str) -> str:
    """Menghapus stopwords dari teks."""
    return " ".join(w for w in text.split() if w not in STOPWORDS_ID)

def tokenize(text: str) -> list:
    """Melakukan tokenisasi teks menjadi daftar kata."""
    return [w for w in text.split() if w]

def stem_text(text: str) -> str:
    """Melakukan stemming untuk teks."""
    return " ".join(stemmer.stem(w) for w in text.split())

print("Fungsi preprocessing siap!")

Fungsi preprocessing siap!


In [ ]:
# ============================================================
# MEMUAT DATA BERITA PERTAMINA
# ============================================================
print("=" * 60)
print("  PREPROCESSING BERITA PERTAMINA")
print("=" * 60)

# Memuat dataset hasil scraping
df_raw = pd.read_csv(file_path, encoding="utf-8") # Menggunakan file_path yang sudah didefinisikan
print(f" Data dimuat: {len(df_raw)} baris, {df_raw.shape[1]} kolom")
print(f"\nKolom yang tersedia: {list(df_raw.columns)}")

# Preview data
df_raw.head(10)

  PREPROCESSING BERITA PERTAMINA
 Data dimuat: 1567 baris, 7 kolom

Kolom yang tersedia: ['Link', 'Judul', 'Isi Berita', 'Status', 'Tag', 'Sentimen', 'Penerbit']


,Link,Judul,Isi Berita,Status,Tag,Sentimen,Penerbit
0,https://money.kompas.com/image/2017/10/03/1815...,Foto : CSR Pertamina Lubricants Kini Fokus ke ...,Diskusi mengenai CSR di industri pelumas berta...,success,Ekonomi,Netral,money.kompas.com
1,https://money.kompas.com/read/2016/06/06/14280...,"Gandeng Dua Bank, Pertamina Lubricants Yakin P...","JAKARTA, KOMPAS.com - Anak perusahaan Pertamin...",success,Kecelakaan Kerja,Positif,money.kompas.com
2,https://money.kompas.com/image/2017/08/03/0900...,"Foto : Cegah Kepunahan, Pertamina Lestarikan T...",Pelepasan Indukan Tuntong Laut oleh tim konser...,success,Bisnis,Positif,money.kompas.com
3,https://money.kompas.com/read/2016/04/11/17345...,Pertamina Menaikkan Ongkos Angkut Minyak Penam...,"BOJONEGORO, KOMPAS.com - Pemerintah terus meng...",success,Migas,Positif,money.kompas.com
4,https://money.kompas.com/image/2017/11/27/1656...,Foto : Ini Dua Fokus CSR Pertamina Patra Niaga...,Direktur Operasional PT Pertamina Patra Niaga ...,success,Akademik,Positif,money.kompas.com
5,https://money.kompas.com/image/2016/11/01/1929...,Foto : Pertamina Tegaskan Kelangkaan BBM Hanya...,"Unduh Kompas.com App untuk berita terkini, aku...",success,BBM,Negatif,money.kompas.com
6,https://nasional.kompas.com/read/2017/06/09/20...,Mantan Dirut Pertamina Transkontinental Diteta...,"JAKARTA, KOMPAS.com - Jaksa Agung Muda Pidana ...",success,Hukum,Netral,nasional.kompas.com
7,https://money.kompas.com/image/2017/03/24/1130...,Foto : Ke Mana Wianda Pusponegoro Setelah Tak ...,"Unduh Kompas.com App untuk berita terkini, aku...",success,Lainnya,Netral,money.kompas.com
8,https://otomotif.kompas.com/read/2017/09/16/16...,"Setelah Tol, Beli BBM Juga Wajib Pakai Uang El...","Jakarta, KompasOtomotif - Selain PT Jasa Marga...",success,BBM,Negatif,otomotif.kompas.com
9,https://otomotif.kompas.com/read/2017/12/13/11...,Pertamax Turbo buat Konsumen Pemilik Supercar,"Nusa Dua, KompasOtomotif - Pertamina kini teng...",success,Kebakaran,Netral,otomotif.kompas.com


In [ ]:
# ============================================================
# PIPELINE PREPROCESSING DENGAN PROGRESS BAR DAN BOILERPLATE REMOVAL
# ============================================================
from tqdm.auto import tqdm

# Mengaktifkan progress_apply() untuk pandas
tqdm.pandas()

print("[→] Menjalankan pipeline preprocessing...")

# 1. Uppercase & Lowercase
print("  [1/8] Uppercase & Lowercase...")
df["text_uppercase"] = df["Isi Berita"].progress_apply(to_uppercase)
df["text_lowercase"] = df["Isi Berita"].progress_apply(to_lowercase)

# 2. Cleaning teks
print("  [2/8] Cleaning teks...")
df["text_clean"] = df["text_lowercase"].progress_apply(clean_text)

# 3. Menghapus boilerplate berita
print("  [3/8] Menghapus boilerplate berita...")
df["text_no_boilerplate"] = df["text_clean"].progress_apply(remove_news_boilerplate)

# 4. Tokenisasi sebelum stopwords
print("  [4/8] Tokenisasi (sebelum stopwords)...")
df["tokens_before_sw"] = df["text_no_boilerplate"].progress_apply(tokenize)
df["jumlah_token_before_sw"] = df["tokens_before_sw"].apply(len)

# 5. Menghapus stopwords
print("  [5/8] Stopwords removal...")
df["text_no_stopwords"] = df["text_no_boilerplate"].progress_apply(remove_stopwords)

# 6. Tokenisasi setelah stopwords
print("  [6/8] Tokenisasi (setelah stopwords)...")
df["tokens_after_sw"] = df["text_no_stopwords"].progress_apply(tokenize)
df["jumlah_token_after_sw"] = df["tokens_after_sw"].apply(len)

# 7. Format tokenisasi untuk Excel
print("  [7/8] Format tokenisasi...")
df["tokenisasi"] = df["tokens_after_sw"].progress_apply(lambda x: " | ".join(x))

# 8. Stemming
print("  [8/8] Stemming (harap tunggu...)")
df["text_stemmed"] = df["text_no_stopwords"].progress_apply(stem_text)
df["tokens_stemmed"] = df["text_stemmed"].progress_apply(tokenize)

print("\n Preprocessing selesai!")

# Tampilkan beberapa hasil preprocessing
df[["Judul", "text_clean", "text_stemmed", "jumlah_token_before_sw", "jumlah_token_after_sw"]].head(10)

[→] Menjalankan pipeline preprocessing...
  [1/8] Uppercase & Lowercase...


  0%|          | 0/1567 [00:00<?, ?it/s]

  0%|          | 0/1567 [00:00<?, ?it/s]

  [2/8] Cleaning teks...


  0%|          | 0/1567 [00:00<?, ?it/s]

...

  [8/8] Stemming (harap tunggu...)


  0%|          | 0/1567 [00:00<?, ?it/s]

  0%|          | 0/1567 [00:00<?, ?it/s]


 Preprocessing selesai!


,Judul,text_clean,text_stemmed,jumlah_token_before_sw,jumlah_token_after_sw
0,Foto : CSR Pertamina Lubricants Kini Fokus ke ...,diskusi mengenai csr industri pelumas bertajuk...,diskusi kena csr industri lumas tajuk cari for...,16,13
1,"Gandeng Dua Bank, Pertamina Lubricants Yakin P...",jakarta kompas com anak perusahaan pertamina y...,anak usaha pertamina pertamina lubricants gand...,130,95
2,"Foto : Cegah Kepunahan, Pertamina Lestarikan T...",pelepasan indukan tuntong laut oleh tim konser...,lepas indu tuntong laut tim konservasi alam pe...,34,31
3,Pertamina Menaikkan Ongkos Angkut Minyak Penam...,bojonegoro kompas com pemerintah terus mengupa...,bojonegoro perintah terus upaya tambah produks...,307,229
4,Foto : Ini Dua Fokus CSR Pertamina Patra Niaga...,direktur operasional pertamina patra niaga abd...,operasional pertamina patra niaga abdul cholid...,38,32
5,Foto : Pertamina Tegaskan Kelangkaan BBM Hanya...,unduh kompas com app untuk berita terkini akur...,unduh app kini akurat percaya arah kamera kode...,17,10
6,Mantan Dirut Pertamina Transkontinental Diteta...,jakarta kompas com jaksa agung muda pidana khu...,jaksa agung muda pidana khusus jaksa agung tet...,145,104
7,Foto : Ke Mana Wianda Pusponegoro Setelah Tak ...,unduh kompas com app untuk berita terkini akur...,unduh app kini akurat percaya arah kamera kode...,17,10
8,"Setelah Tol, Beli BBM Juga Wajib Pakai Uang El...",jakarta kompasotomotif selain jasa marga perse...,kompasotomotif jasa marga pertamina siap laku ...,290,225
9,Pertamax Turbo buat Konsumen Pemilik Supercar,nusa dua kompasotomotif pertamina kini tengah ...,nusa kompasotomotif pertamina tengah upaya hil...,214,166


In [ ]:
# ============================================================
# MENYIMPAN HASIL PREPROCESSING KE GOOGLE DRIVE
# ============================================================
OUTPUT_FILE = "hasil_preprocessing_pertamina.csv"  # Nama file yang ingin disimpan
OUTPUT_PATH = '/content/drive/MyDrive/ProjectA-PBA/' + OUTPUT_FILE  # Path untuk menyimpan ke Google Drive

# Menyimpan hasil preprocessing ke Google Drive
df.to_csv(OUTPUT_PATH, index=False)

print(f"Dataset hasil preprocessing berhasil disimpan sebagai {OUTPUT_PATH}")

Dataset hasil preprocessing berhasil disimpan sebagai /content/drive/MyDrive/ProjectA-PBA/hasil_preprocessing_pertamina.csv
